# Animations over model time, and following a tracked storm

A storm's life cycle spans hundreds of output files, each several
gigabytes. They do not fit in memory together, so skyvista animates them
one frame at a time with `sv.BlenderAnimation`.

This notebook covers:

1. how a `BlenderAnimation` works;
2. a time-lapse of a RAMS storm with a fixed camera;
3. a camera that follows a storm along its tobac track;
4. isolating one tracked cell from its neighbours;
5. per-frame labels: a time stamp and a caption that changes with time.

To keep the run short, each animation here has only a few frames, at a
small size. The full supercell animation, with 345 frames, is
`examples/cm1_storms/outputs/supercell_animation.mp4`.

Run it in the pixi `blender` environment. `01_storm_portrait.ipynb`
explains the basics.

In [ ]:
import sys
from pathlib import Path

# notebook_setup.py sits next to this notebook. Jupyter starts the kernel
# either in this folder or in the repository root (VS Code's default).
for notebook_folder in (Path.cwd(), Path.cwd() / "examples" / "notebooks"):
    if (notebook_folder / "notebook_setup.py").exists():
        sys.path.insert(0, str(notebook_folder))
        break

import numpy as np
import pandas as pd
import xarray as xr
import xarrams

import notebook_setup as setup
import skyvista as sv
from skyvista.cloud_optics import cm1_morrison_hydrometeor_species

## 1. How a `BlenderAnimation` works

You give it three things:

- `times`: the model times of the frames.
- `build_frame(time)`: a function that loads one time and returns a
  `Scene`. Only this function touches the data.
- `config`: the export settings, as for a still.

`run()` then goes through five stages:

1. **Export.** Several processes call `build_frame` in parallel. Each frame
   becomes a small bundle in `frames/`.
2. **Assemble.** The frame bundles merge into one animation bundle in
   `bundle/`. Each frame's data files become numbered files of one
   sequence.
3. **Render.** Blender renders the animation bundle. Several Blender
   processes can share the frames.
4. **Compose.** A `sv.compose.Figure` draws labels on every frame, in
   `composed/`.
5. **Encode.** ffmpeg writes an MP4 video.

Every stage skips work already on disk. An interrupted run therefore
resumes where it stopped when it is run again.

**One coordinate system.** Every frame must map model coordinates to
Blender in the same way. Set `config.transform.origin_shift`, the model
point that becomes Blender's origin, to a fixed point.

## 2. A time-lapse with a fixed camera

Four RAMS output times, 15 min apart, from towering cumulus to mature
storm. Without a camera path, the camera frames the frame with the
largest cloud, so the mature storm fits. Frames without cloud are
ignored for that choice.

`build_rams_frame` reads one output file with `xarrams`, as in notebook
01. The model time of each file comes from its name.

In [ ]:
# RAMS hydrometeor mixing ratios (R?P), number concentrations (C?P) and
# the base-state air density.
RAMS_CLOUD_VARIABLES = [
    "RCP", "RDP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP",
    "CCP", "CDP", "CRP", "CPP", "CSP", "CAP", "CGP", "CHP",
    "DN0",
]
# Levels kept: from the ground to the highest cloud (m).
LOWEST_HEIGHT, TOP_HEIGHT = 0.0, 16500.0
# RAMS output names count model time from this date.
RUN_START = pd.Timestamp("2000-01-01")
# Frame size of the preview animations (px).
ANIMATION_RESOLUTION = (640, 360)

# Four output times, 50 to 95 min, keyed by model time (s).
timelapse_minutes = [50, 65, 80, 95]
timelapse_files = [
    setup.RAMS_STORM_OUTPUT_DIR / f"a-L-2000-01-01-{minutes // 60:02d}{minutes % 60:02d}00-g1.h5"
    for minutes in timelapse_minutes
]
files_by_time = {
    (xarrams.get_datetime_from_rams_filename(path) - RUN_START).total_seconds(): path
    for path in timelapse_files
}


def build_rams_frame(time_seconds):
    """The scene at one time: the clouds of that output file."""
    frame_output = xarrams.read_rams_output(
        [files_by_time[time_seconds]], keep_vars=RAMS_CLOUD_VARIABLES, silent=True
    )
    frame_clouds = frame_output.isel(time=0).sel(z=slice(LOWEST_HEIGHT, TOP_HEIGHT)).load()
    return sv.Scene().add_clouds(frame_clouds, microphysics="rams")


timelapse_config = sv.BlenderExportConfig.for_clouds(quality="preview")
timelapse_config.render.resolution = ANIMATION_RESOLUTION
# A low camera south-east of the storm, the sun from the north-east.
timelapse_config.camera.azimuth_deg = -55.0
timelapse_config.camera.elevation_deg = -8.0
timelapse_config.camera.distance_factor = 1.1
timelapse_config.world.sun_azimuth_deg = 15.0
timelapse_config.world.sun_elevation_deg = 30.0
# One fixed origin for every frame: the domain center, at the ground.
timelapse_config.transform.origin_shift = (0.0, 0.0, 0.0)

The figure stamps each frame with its model time. `TimeStamp` reads the
frame's time from the animation. Its `formatter` turns seconds into the
text.

`Panel(lambda frame: frame.render_path, ...)` makes the panel's image a
function of the frame: each composed frame shows its own render.

In [ ]:
timelapse_figure = sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            lambda frame: frame.render_path,
            [
                sv.compose.TimeStamp(
                    formatter=lambda seconds: f"t = {seconds / 60:.0f} min", size=0.07
                )
            ],
        )
    ]
)

timelapse = sv.BlenderAnimation(
    setup.WORK_DIR / "rams_timelapse",
    times=sorted(files_by_time),
    build_frame=build_rams_frame,
    config=timelapse_config,
    fps=1.0,
)
timelapse_video = timelapse.run(figure=timelapse_figure, export_workers=4)

In [ ]:
# The composed frames as one contact sheet, 2 x 2.
timelapse_sheet_path = setup.WORK_DIR / "rams_timelapse_sheet.png"
sv.compose.Figure(
    panels=[sv.compose.Panel(path) for path in sorted(timelapse.composed_dir.glob("frame_*.png"))],
    n_columns=2,
).render(timelapse_sheet_path)
setup.show_image(timelapse_sheet_path, width=900)

## 3. A camera that follows a tracked storm

The second example is a CM1 supercell. The domain also holds other
storms, so the camera follows this one along its *tobac* track. tobac is
a Python package that finds storm features in each output and links them
into tracks over time.

The tracking output of this run is in its `derived/` folder:

- `_all_tracks.csv`: every feature of every cell at every time. `cell` is
  the track number. `hdim_1` and `hdim_2` are the feature's position as
  fractional grid indices along y and x. `num` is its size in grid points.
- `tracks.csv`: the features of the one selected cell, the right-moving
  supercell.

Around 59 min the storm splits into a right mover and a left mover. The
right mover is a new cell from then on. Before the split, tobac relabels
the parent storm several times. The track below therefore takes the
largest feature at each time before the split, and the selected cell
after it.

In [ ]:
derived_dir = setup.CM1_SUPERCELL_RUN / "derived"
all_features = pd.read_csv(derived_dir / "_all_tracks.csv", parse_dates=["time"])
selected_features = pd.read_csv(derived_dir / "tracks.csv", parse_dates=["time"])

# Model time (s) of each feature; the tables count from 2000-01-01 00:00.
all_features["seconds"] = (all_features["time"] - RUN_START).dt.total_seconds()
selected_cell = int(selected_features["cell"].iloc[0])
split_seconds = float(
    all_features.loc[all_features["cell"] == selected_cell, "seconds"].min()
)
print(f"selected cell {selected_cell}, from t = {split_seconds / 60:.1f} min")
all_features[["time", "cell", "hdim_1", "hdim_2", "num"]].head()

`sv.Track` holds a position over time and interpolates between the
tracked times. Its `smoothing` (s) is the width of a Gaussian average
over time. The hand-off from the parent storm to the right mover jumps
several kilometers; smoothing over 5 min turns that jump into a glide.

In [ ]:
# The parent storm (largest feature) before the split; the right mover after.
before_split = all_features[all_features["seconds"] < split_seconds]
parent_storm = before_split.loc[before_split.groupby("seconds")["num"].idxmax()]
right_mover = all_features[
    (all_features["seconds"] >= split_seconds) & (all_features["cell"] == selected_cell)
]
storm_features = pd.concat([parent_storm, right_mover]).sort_values("seconds")

# Fractional grid indices to meters, with the run's grid coordinates.
with xr.open_dataset(setup.CM1_SUPERCELL_RUN / "output" / "cm1out_000002.nc") as grid_file:
    grid_x = grid_file["xh"].values
    grid_y = grid_file["yh"].values
storm_x = np.interp(storm_features["hdim_2"], np.arange(len(grid_x)), grid_x)
storm_y = np.interp(storm_features["hdim_1"], np.arange(len(grid_y)), grid_y)

storm_track = sv.Track(
    times=storm_features["seconds"].to_numpy(), x=storm_x, y=storm_y, smoothing=300.0
)
for seconds in (600.0, 3600.0, 7200.0):
    track_x, track_y = storm_track.position(seconds)
    print(f"t = {seconds / 60:3.0f} min: storm at ({track_x / 1e3:.0f}, {track_y / 1e3:.0f}) km")

A `CameraPath` is a list of `CameraKeyframe`s in model time. With
`target=storm_track`, each keyframe places the camera relative to the
storm:

- `azimuth_deg`: the direction from the storm to the camera.
- `horizontal_distance`: the camera's distance from the storm (m).
- `height`: the camera's height above the ground (m).
- `target_height`: the height above the ground of the point it aims at,
  directly over the storm.
- `lens_mm`: the focal length.

A keyframe can leave a value out. That value is then interpolated
through the keyframes that do set it. Between keyframes, each value is interpolated with a monotone cubic
curve, which never overshoots. The camera here pulls back and rises as
the storm grows, and swings 30 degrees around it.

In [ ]:
supercell_camera_path = sv.CameraPath(
    [
        # A few cumulus towers, about 3 km tall.
        sv.CameraKeyframe(
            300.0, azimuth_deg=-75.0, horizontal_distance=18.0e3,
            height=1000.0, target_height=2500.0, lens_mm=28.0,
        ),
        # Towering cumulus reaching 13 km.
        sv.CameraKeyframe(1200.0, horizontal_distance=32.0e3, height=1500.0, target_height=6000.0),
        # The split: a 21 km cumulonimbus with a spreading anvil.
        sv.CameraKeyframe(3600.0, horizontal_distance=50.0e3, target_height=8000.0),
        # The mature right mover.
        sv.CameraKeyframe(7180.0, azimuth_deg=-45.0, horizontal_distance=60.0e3),
    ],
    target=storm_track,
)

## 4. Isolating the tracked cell

After the split, the left mover drifts north and would share the frame.
Its anvil merges with the right mover's, so no gap in the cloud separates
them. The tracking gives the dividing line instead.

`storm_keep_weight` below gives every column a weight between 0 and 1:

1. It takes the tracked features at the time nearest the frame.
2. It finds the selected cell and its *competitors*: the other cells at
   least 10% of its size. Smaller ones are fragments of the same storm.
3. Each column goes to the nearer of the selected cell and its nearest
   competitor. The dividing line runs halfway between them.
4. The weight is 1 on the selected cell's side and 0 on the other side.
   It changes linearly across a band 10 km wide, so the cut anvil thins
   out instead of ending in a flat wall.

`add_clouds(weight="storm_weight")` multiplies the cloud's extinction by
the dataset variable `storm_weight`. The particle sizes, and so the
colors and the scattering, stay as they are. Removed cloud also stops
counting for the size of the exported volume.

Removing the left mover in one frame would look like a jump cut. Instead,
`build_supercell_frame` fades it out over the 10 min after the split.

In [ ]:
# Competing cells smaller than this fraction of the selected cell are
# fragments of the same storm.
COMPETING_CELL_MINIMUM_SIZE_FRACTION = 0.1
# Width of the fade between the selected cell and its competitor (m).
ISOLATION_TAPER_WIDTH = 10.0e3


def feature_position(feature):
    """A tobac feature's (x, y) in meters, from its fractional grid indices."""
    feature_x = float(np.interp(feature["hdim_2"], np.arange(len(grid_x)), grid_x))
    feature_y = float(np.interp(feature["hdim_1"], np.arange(len(grid_y)), grid_y))
    return feature_x, feature_y


def storm_keep_weight(frame_clouds, time_seconds):
    """
    A (x, y) weight: 1 on the selected cell's side, 0 on its competitor's.

    Each column goes to the nearer of the selected cell and its nearest
    competitor, at the tracked time nearest the frame. The split is
    softened into a linear fade ISOLATION_TAPER_WIDTH wide.
    """
    # The tracked features at the tracked time nearest this frame.
    time_differences = (all_features["seconds"] - time_seconds).abs()
    nearest_tracked_seconds = all_features["seconds"].iloc[time_differences.argmin()]
    features_now = all_features[all_features["seconds"] == nearest_tracked_seconds]

    # The selected cell, and the other cells large enough to compete.
    selected_feature = features_now[features_now["cell"] == selected_cell].iloc[0]
    is_other_cell = features_now["cell"] != selected_cell
    is_large = features_now["num"] >= COMPETING_CELL_MINIMUM_SIZE_FRACTION * selected_feature["num"]
    competing_features = features_now[is_other_cell & is_large]

    # Distance of every column to the selected cell (a DataArray over x, y).
    selected_x, selected_y = feature_position(selected_feature)
    distance_to_selected = np.hypot(frame_clouds["x"] - selected_x, frame_clouds["y"] - selected_y)

    # Distance of every column to the nearest competitor.
    distance_to_competitor = xr.full_like(distance_to_selected, np.inf)
    for _, competing_feature in competing_features.iterrows():
        competitor_x, competitor_y = feature_position(competing_feature)
        distance_to_this_competitor = np.hypot(
            frame_clouds["x"] - competitor_x, frame_clouds["y"] - competitor_y
        )
        distance_to_competitor = np.minimum(distance_to_competitor, distance_to_this_competitor)

    # 1 on the selected cell's side, 0 on the competitor's, linear between.
    distance_advantage = distance_to_competitor - distance_to_selected
    return (0.5 + distance_advantage / ISOLATION_TAPER_WIDTH).clip(0.0, 1.0)

In [ ]:
# Morrison mixing ratios (kg/kg), number concentrations (#/kg) and the
# air density (kg/m^3), as in notebook 01.
CM1_CLOUD_VARIABLES = ["qc", "qr", "qi", "qs", "qg", "nci", "ncr", "ncs", "ncg", "rho"]
# The region the storm crosses during the animation (m).
SUPERCELL_X_RANGE, SUPERCELL_Y_RANGE = (80.0e3, 320.0e3), (50.0e3, 230.0e3)
# Morrison settings of this run's namelist: ndcnst = 300 cm^-3, ihail = 1.
supercell_species = cm1_morrison_hydrometeor_species(
    droplet_number_concentration=300.0e6, graupel_density=900.0
)
# The left mover fades out over this long after the split (s).
FADE_DURATION_SECONDS = 600.0
# Name of the weight variable that add_clouds multiplies the cloud by.
STORM_WEIGHT_VARIABLE = "storm_weight"


def cm1_output_file(run_dir, time_seconds):
    """
    The CM1 output file nearest a model time.

    CM1 numbers its files from 1 at t = 0, one per output interval. The
    interval is the time of the second file.
    """
    with xr.open_dataset(run_dir / "output" / "cm1out_000002.nc") as second_file:
        output_interval = float(second_file["time"].values[0])
    file_number = int(round(time_seconds / output_interval)) + 1
    return run_dir / "output" / f"cm1out_{file_number:06d}.nc"


def build_supercell_frame(time_seconds):
    """
    The scene at one time: the storm's cloud, with the left mover fading.

    After the split, the keep weight (1 on the right mover's side, 0 on
    the left mover's) is blended toward 1 by the fade:
    ``1 - fade * (1 - weight)``. Fade 0 removes nothing; fade 1 removes
    the left mover entirely.
    """
    # The storm's region at this time: hydrometeors, renamed to x, y, z.
    frame_output = xarrams.read_cm1_output(
        [cm1_output_file(setup.CM1_SUPERCELL_RUN, time_seconds)],
        keep_vars=CM1_CLOUD_VARIABLES,
        silent=True,
    )
    frame_clouds = (
        frame_output.isel(time=0)
        .rename({"xh": "x", "yh": "y", "zh": "z"})
        .sel(x=slice(*SUPERCELL_X_RANGE), y=slice(*SUPERCELL_Y_RANGE))
        .load()
    )

    # Before the split there is nothing to fade.
    fade = float(np.clip((time_seconds - split_seconds) / FADE_DURATION_SECONDS, 0.0, 1.0))
    if fade == 0.0:
        return sv.Scene().add_clouds(
            frame_clouds, species=supercell_species, vertical_spacing=125.0
        )

    # After it, the left mover fades out through the cloud weight.
    keep_weight = storm_keep_weight(frame_clouds, time_seconds)
    frame_clouds[STORM_WEIGHT_VARIABLE] = 1.0 - fade * (1.0 - keep_weight)
    return sv.Scene().add_clouds(
        frame_clouds,
        species=supercell_species,
        vertical_spacing=125.0,
        weight=STORM_WEIGHT_VARIABLE,
    )

## 5. Per-frame labels

Any text in the figure can be a function of the frame. The function
receives a `FrameContext`, which holds the frame's index and model time.
`phase_caption` below names the stage of the storm's life.

In [ ]:
def phase_caption(frame):
    """What the storm is doing at this frame."""
    if frame.time_seconds < split_seconds:
        return "Parent storm"
    if frame.time_seconds < split_seconds + FADE_DURATION_SECONDS:
        return "The storm splits; the left mover is faded out"
    return "Right mover"


supercell_figure = sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            lambda frame: frame.render_path,
            [
                sv.compose.Text("Right-moving supercell (CM1, 250 m grid)", size=0.06),
                sv.compose.Text(phase_caption, size=0.05, offset=(0.0, -0.085)),
                sv.compose.TimeStamp(anchor="lower left", size=0.06),
            ],
        )
    ]
)

supercell_config = sv.BlenderExportConfig.for_clouds(quality="preview")
supercell_config.render.resolution = ANIMATION_RESOLUTION
supercell_config.world.sun_azimuth_deg = -150.0
supercell_config.world.sun_elevation_deg = 30.0
# One fixed origin: the domain's south-west corner, at the ground.
supercell_config.transform.origin_shift = (0.0, 0.0, 0.0)

# Six times across the life cycle: 5, 25, 55, 65, 90 and 120 min.
supercell_times = [300.0, 1500.0, 3300.0, 3900.0, 5400.0, 7180.0]
supercell_animation = sv.BlenderAnimation(
    setup.WORK_DIR / "supercell_preview",
    times=supercell_times,
    build_frame=build_supercell_frame,
    config=supercell_config,
    camera_path=supercell_camera_path,
    fps=1.0,
)
supercell_video = supercell_animation.run(figure=supercell_figure, export_workers=6)

In [ ]:
# The composed frames as one contact sheet, 2 x 3.
supercell_sheet_path = setup.WORK_DIR / "supercell_preview_sheet.png"
sv.compose.Figure(
    panels=[
        sv.compose.Panel(path)
        for path in sorted(supercell_animation.composed_dir.glob("frame_*.png"))
    ],
    n_columns=2,
).render(supercell_sheet_path)
setup.show_image(supercell_sheet_path, width=1000)

## From preview to the full animation

`examples/cm1_storms/animate_supercell.py` makes the full animation: one
frame every 20 s of model time from 5 min to 2 h, 345 frames at 1280 x 720
and 16 samples. On a 64-core machine:

- the export took 21 min with 12 processes;
- the render took about 5 h with 4 Blender processes of 16 threads each.

Useful settings for long runs:

- `run(export_workers=...)`: each export process holds one frame's data in
  memory. A 400 x 400 x 330 RAMS grid needs 10-20 GB per process.
- `run(render_processes=...)`: several Blender processes share the
  frames. This uses many cores better than one large process.
- An interrupted run resumes when it is started again with the same
  arguments.